# 002 · Counting with Counter

> **Standard Library › Data Structures** · Python 3.14

"How many of each?" is one of the most common questions in programming. `collections.Counter` is a `dict` subclass purpose-built for it: build counts from any iterable in one call, get the top N, and add or subtract whole tallies like multisets.

**In this notebook you will learn:**

- how `Counter` replaces `Dictionary<T, int>` loops and `GroupBy(...).Count()` queries
- `most_common`, `total`, and `elements`
- updating counts with `update` / `subtract`
- multiset arithmetic with `+`, `-`, `&`, `|`
- why missing keys return `0` instead of throwing

## .NET mental model

| .NET | Python |
|---|---|
| `Dictionary<T, int>` + `TryGetValue` / `++` loop | `Counter(items)` |
| `items.GroupBy(x => x).ToDictionary(g => g.Key, g => g.Count())` | `Counter(items)` |
| `items.CountBy(x => x)` (.NET 9) | `Counter(items)` |
| `.OrderByDescending(kv => kv.Value).Take(n)` | `counter.most_common(n)` |
| `.Values.Sum()` | `counter.total()` |
| `dict[key]` → `KeyNotFoundException` | `counter[key]` → `0` |

A `Counter` **is a** `dict` (`Counter` subclasses `dict`), so everything you know about dictionaries *(see Syntax › Collections › 003 · Dictionaries)* still applies.

## Building a Counter

```csharp
// C#
var counts = new Dictionary<string, int>();
foreach (var w in words)
    counts[w] = counts.GetValueOrDefault(w) + 1;

// or LINQ
var counts2 = words.GroupBy(w => w).ToDictionary(g => g.Key, g => g.Count());
```

In [ ]:
from collections import Counter

words = "the cat and the hat and the bat".split()
counts = Counter(words)
print(counts)

print(Counter("mississippi"))                 # any iterable — here, characters
print(Counter({"apples": 3, "pears": 1}))     # from a mapping
print(Counter(red=2, blue=5))                 # from keyword arguments

Counting a derived key? Feed a generator expression — the equivalent of `CountBy(selector)`:

In [ ]:
orders = [
    {"id": 1, "status": "shipped"},
    {"id": 2, "status": "pending"},
    {"id": 3, "status": "shipped"},
    {"id": 4, "status": "cancelled"},
]
by_status = Counter(o["status"] for o in orders)
print(by_status)

words = ["apple", "fig", "kiwi", "banana", "plum"]
print(Counter(len(w) for w in words))         # how many words of each length

## Missing keys return 0

Unlike a plain `dict` (or `Dictionary<TKey, TValue>`), looking up a missing key returns `0` — no `KeyError`, and the key is **not** inserted. That makes `counter[x] += 1` work without any setup.

In [ ]:
c = Counter(["a", "b", "a"])
print(c["a"], c["zzz"])      # zzz was never counted -> 0
print("zzz" in c)            # still not a key

c["new"] += 1                # no KeyError, no "if key in dict" dance
print(c)

⚠️ **Gotcha:** setting a count to zero does **not** remove the key. Use `del` to remove it — and remember `len(counter)` counts **keys**, not items.

In [ ]:
c = Counter(a=2, b=1)
c["b"] = 0
print(c, len(c))     # 'b' still there, len is 2
del c["b"]
print(c, len(c))
print(c.total())     # sum of all counts (3.10+) — use this for "number of items"

## `most_common`

`most_common(n)` returns the `n` highest counts as `(key, count)` tuples, highest first. Ties keep the order in which the keys were first seen. Without `n` you get everything, sorted.

```csharp
// C#
var top = counts.OrderByDescending(kv => kv.Value).Take(3);
```

In [ ]:
text = "the cat and the hat and the bat sat on the mat"
counts = Counter(text.split())

print(counts.most_common(2))
for word, n in counts.most_common(3):
    print(f"{word:>5}: {'#' * n}")

print(counts.most_common()[-1])   # least common (last of the full list)

## `update` and `subtract`

`update` **adds** counts (unlike `dict.update`, which replaces values); `subtract` subtracts them, keeping zero and negative results.

In [ ]:
stock = Counter(apples=5, pears=2)
stock.update(["apples", "kiwis", "kiwis"])   # from an iterable…
stock.update({"pears": 10})                  # …or a mapping — counts are ADDED
print(stock)

stock.subtract({"apples": 8, "kiwis": 1})
print(stock)                                 # negatives are allowed

## `elements`

`elements()` expands the counter back into an iterator that repeats each key as many times as its count (ignoring counts ≤ 0) — handy for rebuilding a "bag" of items.

In [ ]:
bag = Counter(x=3, y=1, z=0, w=-2)
print(list(bag.elements()))

## Multiset arithmetic

Counters support `+`, `-`, `&` (intersection = min) and `|` (union = max). These operators return a **new** Counter and **drop** zero and negative counts — which makes them behave like multisets.

In [ ]:
a = Counter(apples=3, pears=1)
b = Counter(apples=1, pears=2, kiwis=1)

print("a + b:", a + b)    # add
print("a - b:", a - b)    # subtract, keep only positive results
print("a & b:", a & b)    # min of each
print("a | b:", a | b)    # max of each

**Tip:** "do I have enough ingredients?" — subtract the recipe from the pantry, or use the comparison operators, which compare as multisets (3.10+):

In [ ]:
pantry = Counter(eggs=6, flour=2, milk=1)
recipe = Counter(eggs=2, flour=1, milk=2)

missing = recipe - pantry
print("missing:", missing)
print("can cook:", recipe <= pantry)
print("can cook half batch:", Counter(eggs=1, flour=1) <= pantry)

⚠️ **Gotcha:** `c1 - c2` (operator) and `c1.subtract(c2)` (method) differ: the operator drops non-positive counts and returns a new Counter; the method mutates in place and keeps negatives. Use unary `+c` to strip zero/negative entries from an existing counter.

In [ ]:
c = Counter(a=3, b=1)
c.subtract(Counter(a=1, b=2))
print(c)        # b is -1
print(+c)       # keeps only positive counts

## A typical use: word frequencies

In [ ]:
import re

review = """Great product. Great price. The product arrived late,
but the support team was great and the price was fair."""

words = re.findall(r"[a-z']+", review.lower())
stop = {"the", "and", "was", "but"}
freq = Counter(w for w in words if w not in stop)
print(freq.most_common(3))
print(f"{freq.total()} words, {len(freq)} distinct")

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Count items | `GroupBy(x => x).ToDictionary(..., g => g.Count())` / `CountBy` | `Counter(items)` |
| Count by key | `CountBy(selector)` (.NET 9) | `Counter(f(x) for x in items)` |
| Increment | `d[k] = d.GetValueOrDefault(k) + 1` | `c[k] += 1` |
| Missing key | `KeyNotFoundException` | `0` |
| Top N | `OrderByDescending(v).Take(n)` | `c.most_common(n)` |
| Total items | `d.Values.Sum()` | `c.total()` |
| Add counts | loop | `c.update(items)` / `c1 + c2` |
| Subtract counts | loop | `c.subtract(items)` / `c1 - c2` |
| Expand to items | `SelectMany(kv => Enumerable.Repeat(kv.Key, kv.Value))` | `c.elements()` |
| Multiset subset | — | `c1 <= c2` |

## Exercises

1. Find the most common first letter among `["alice", "bob", "anna", "carol", "adam", "bill"]`.
2. Check whether two strings are anagrams of each other (ignoring spaces and case) using `Counter`.
3. Two warehouses report stock as Counters. Compute the combined stock, the items only the first warehouse can supply more of, and the minimum each warehouse has in common.
4. Given a list of sensor readings, print the readings that appear **more than once**, sorted.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from collections import Counter

# 1
names = ["alice", "bob", "anna", "carol", "adam", "bill"]
print(Counter(n[0] for n in names).most_common(1))

# 2
def is_anagram(a: str, b: str) -> bool:
    norm = lambda s: Counter(s.replace(" ", "").casefold())
    return norm(a) == norm(b)

print(is_anagram("Dormitory", "dirty room"), is_anagram("abc", "abd"))

# 3
w1 = Counter(bolts=50, nuts=20, screws=5)
w2 = Counter(bolts=10, nuts=30, washers=40)
print("combined:", w1 + w2)
print("w1 surplus:", w1 - w2)
print("common min:", w1 & w2)

# 4
readings = [21, 23, 21, 19, 23, 23, 25]
print(sorted(r for r, n in Counter(readings).items() if n > 1))
```

</details>